# 第 13 章 エンドツーエンドの実例

生データの前処理から、3 つのモデルの比較・評価までを 1 本のパイプラインに通します。

対応する記事: [第 13 章 エンドツーエンドの実例（Polyglot Notebook（F#） の言語版）](../../../docs/article/grokking-machine-learning/fsharp/ch13.md)

実装本体: `apps/grokking-ml-fsharp/src/`

## セットアップ

実装本体（`../src/GrokkingMl/`）を `#load` で読み込みます。**ノートブックにコードを複製せず、記事と同じ実装をそのまま使います。**

VS Code の [Polyglot Notebooks 拡張](https://marketplace.visualstudio.com/items?itemName=ms-dotnettools.dotnet-interactive-vscode) で開くか、Jupyter に .NET Interactive カーネルを登録して実行します。

```bash
dotnet tool install -g Microsoft.dotnet-interactive
dotnet interactive jupyter install
jupyter lab notebooks/
```

In [1]:
#load "../src/GrokkingMl/Ch06LogisticRegression.fs"
#load "../src/GrokkingMl/Ch07Metrics.fs"
#load "../src/GrokkingMl/Ch09DecisionTrees.fs"
#load "../src/GrokkingMl/Ch12Ensembles.fs"
#load "../src/GrokkingMl/Ch13EndToEnd.fs"

open GrokkingMl.Ch13EndToEnd

## 生データ

「40 歳未満かつ収入 400 超なら購入」という規則に従う擬似データを作ります。**9 行に 1 行は年齢が欠損** しています。実務のデータは、そのままではモデルに渡せません。

In [2]:
let rng = System.Random(1)
let cities = [ "tokyo"; "osaka"; "kyoto" ]

let rows: Row list =
    List.init 40 (fun i ->
        let age = rng.Next(18, 71)
        let income = rng.Next(200, 901)

        Map.ofList
            [ "age", (if i % 9 = 0 then "" else string age)
              "income", string income
              "city", cities[rng.Next(List.length cities)]
              "bought", (if age < 40 && income > 400 then "yes" else "no") ])

rows |> List.truncate 5 |> List.iter (printfn "%A")

map [("age", ""); ("bought", "no"); ("city", "osaka"); ("income", "277")]

map [("age", "58"); ("bought", "no"); ("city", "osaka"); ("income", "660")]

map [("age", "36"); ("bought", "yes"); ("city", "tokyo"); ("income", "861")]

map [("age", "52"); ("bought", "no"); ("city", "tokyo"); ("income", "220")]

map [("age", "34"); ("bought", "yes"); ("city", "kyoto"); ("income", "893")]

## 前処理

3 つの処理を通します。それぞれ「やらないとどうなるか」が明確です。

| 処理 | やらないとどうなるか |
| :--- | :--- |
| 欠損補完（中央値） | 学習が落ちる。平均だと 1 件の外れ値が全欠損を汚染する |
| 正規化 | 値の大きい特徴量（収入）だけが効く |
| One-Hot | カテゴリに存在しない大小関係が生まれる |

In [3]:
let dataset = buildDataset "bought" rows

printfn "特徴量: %A" dataset.FeatureNames
printfn "陽性 %d 件 / 全 %d 件" (List.sum dataset.Labels) (List.length dataset.Labels)
printfn ""

List.zip dataset.Points dataset.Labels
|> List.truncate 5
|> List.iter (fun (point, label) ->
    printfn "%A → %d" (point |> List.map (sprintf "%.3f")) label)

特徴量: 

["age"; "income"; "city=kyoto"; "city=osaka"; "city=tokyo"]

陽性 

11

 件 / 全 

...

 → 

0

["0.308"; "0.997"; "1.000"; "0.000"; "0.000"]

 → 

1

## 欠損補完と正規化を個別に確かめる

**平均ではなく中央値を使うのは、外れ値に引きずられないため** です。`[1, 2, 3, 1000]` の平均は 251.5 ですが、中央値は 2.5 です。

In [4]:
printfn "欠損補完: %A" (imputeMissing [ Some 1.0; Some 2.0; Some 3.0; Some 1000.0; None ])
printfn "正規化  : %A" (normalize [ 10.0; 20.0; 30.0 ])
printfn "定数列  : %A ← 0 除算しない" (normalize [ 5.0; 5.0; 5.0 ])
printfn "One-Hot : %A" (oneHot [ "b"; "a"; "b" ])

欠損補完: 

[1.0; 2.0; 3.0; 1000.0; 2.5]

正規化  : 

[0.0; 0.5; 1.0]

定数列  : 

[0.0; 0.0; 0.0]

 ← 0 除算しない

One-Hot : 

([[0.0; 1.0]; [1.0; 0.0]; [0.0; 1.0]], ["a"; "b"])

## 3 つのモデルを同じ土俵で比較する

第 6 章のロジスティック回帰、第 9 章の決定木、第 12 章の AdaBoost を、第 7 章の指標で評価します。

**正解率だけを見ていると差を見落とします。** 指標ごとに順位が変わりうることに注目してください。

In [5]:
let evaluations = runPipeline "bought" rows

printfn "%-12s %8s %8s %8s %8s %8s" "モデル" "正解率" "適合率" "再現率" "F1" "AUC"

for e in evaluations do
    printfn "%-12s %8.3f %8.3f %8.3f %8.3f %8.3f" e.Name e.Accuracy e.Precision e.Recall e.F1 e.Auc

printfn ""
printfn "F1 が最良のモデル: %s" (bestByF1 evaluations).Name

モデル         

     正解率

     適合率

     再現率

      F1

...

   0.600

   0.750

   0.800

F1 が最良のモデル: 

logistic

## 試してみる: 評価は揺れる

テストデータは 12 件しかありません。**1 件の当たり外れが正解率を 0.083 動かします。**

分割のシードを変えて、どれくらい結果が揺れるか見てみましょう。「モデル A のほうが 3% 良い」という報告が、この規模では意味を持たないことが分かります。実務では交差検証で複数の分割を試し、平均と分散を見ます。

In [6]:
printfn "%6s %10s %8s" "シード" "テスト件数" "陽性数"

for seed in 0..3 do
    let split = splitDataset 0.3 seed dataset
    printfn "%6d %10d %8d" seed (List.length split.TestLabels) (List.sum split.TestLabels)

   シード

     テスト件数

     陽性数

     0

        12

...

        12

       6

     3

        12

       2